# 🚀 Few-Shot Cross-Domain 3D Multi-Organ Segmentation
## Fault-Tolerant Benchmark Runner with Hugging Face Hub Auto-Sync

### 🛡️ Built-in Free-Tier & Disconnection Protections:
1. **Hugging Face Hub Cloud Sync:** Checkpoints (`checkpoint_latest.pth`, `checkpoint_best.pth`) and CSV result tables are automatically pushed to your private Hugging Face repository.
2. **Seamless Cloud Auto-Resume:** If your Colab or Kaggle session disconnects, times out, or gets preempted, re-running this notebook automatically pulls your latest weights from Hugging Face and resumes from the exact epoch without losing progress!
3. **Persistent Result Caching:** Completed experimental regimes ($k=1, 3, 5, 10$) are logged and skipped automatically upon resume to save your GPU quota.
4. **Complete Artifact Export:** Automatically bundles all metrics, per-case details, and convergence curves into a single downloadable `results_bundle.zip`.

### Step 1: Environment & GPU Verification

In [ ]:
# Install lightweight dependencies
!pip install -q nibabel huggingface_hub scipy tqdm pandas matplotlib

import torch
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ GPU not detected. Please enable GPU in Runtime -> Change runtime type (Colab) or Accelerator -> GPU P100 (Kaggle).")

### Step 2: Configure Hugging Face Cloud Storage for Checkpoints
Enter your **Hugging Face Write Token** when prompted below, or set it via Kaggle Secrets.

In [ ]:
import os, getpass

# -------------------------------------------------------------------------
# HUGGING FACE CREDENTIALS & PRIVATE CLOUD REPO
# -------------------------------------------------------------------------
HF_USERNAME = "Pronob002"
HF_REPO = f"{HF_USERNAME}/hybrid-swin-unet-checkpoints"

# Read from Kaggle/Colab environment or prompt securely
if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"]:
    HF_TOKEN = os.environ["HF_TOKEN"]
else:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = getpass.getpass("Enter your Hugging Face Write Token: ")

os.environ["HF_TOKEN"] = HF_TOKEN
print(f"✅ Target Hugging Face Cloud Repo: https://huggingface.co/{HF_REPO}")

### Step 3: Load Workspace Code
Clones or navigates to the repository.

In [ ]:
import os, sys
if not os.path.exists("src"):
    !git clone https://github.com/pronob002/Hybrid_Swin_UNet.git
    %cd Hybrid_Swin_UNet

sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

### Step 4: Run 5-Shot Adaptation Benchmark
Runs the **Proposed Hybrid Swin-UNet** and **3D U-Net Baseline** with automatic cloud checkpointing and auto-resume.

In [ ]:
# 1. Hybrid Swin-UNet (Proposed)
!python train.py --model hybrid_swin --shots 5 --epochs 50 --eval_cases 10 --seed 42 --hf_repo $HF_REPO --hf_token $HF_TOKEN

# 2. 3D U-Net Baseline
!python train.py --model unet3d --shots 5 --epochs 50 --eval_cases 10 --seed 42 --hf_repo $HF_REPO --hf_token $HF_TOKEN

### Step 5: Multi-Regime Benchmark ($k=1, 3, 5, 10$ Shots & Multi-Seed Loop)
This loop automatically checks the Hugging Face result cache and skips already-completed runs if interrupted.

In [ ]:
from scripts.train_fewshot import run_experiment

regimes = [1, 3, 5, 10]
models = ["hybrid_swin", "unet3d"]
seeds = [42, 123, 456]

for seed in seeds:
    for m in models:
        for k in regimes:
            print(f"\n{'='*70}")
            print(f">>> Running {m.upper()} | {k}-Shot Adaptation | Seed: {seed}")
            print(f"{'='*70}")
            run_experiment(
                model_type=m,
                k_shots=k,
                num_epochs=50,
                seed=seed,
                eval_cases=10,
                checkpoint_base_dir="checkpoints",
                results_dir="results",
                hf_repo=HF_REPO,
                hf_token=HF_TOKEN,
                force_rerun=False  # Automatically skips completed experiments!
            )

### Step 6: Post-Benchmark Analysis & Publication Table Generation

In [ ]:
# Run statistical tests (Wilcoxon p-values) and generate CBM tables & figures
!python scripts/analyze_results.py

import pandas as pd
csv_path = "results/benchmark_summary.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\n--- Completed Benchmark Summary ---")
    display(df)

### Step 7: Create 1-Click Downloadable Zip Bundle
Packages `results/`, `checkpoints/`, and `figures/` into `results_bundle.zip` for instant download.

In [ ]:
import shutil, os
from huggingface_hub import HfApi

!zip -r results_bundle.zip results checkpoints figures
print("\n✅ [SUCCESS] Created results_bundle.zip!")

# Also push the complete zip bundle to Hugging Face
if os.path.exists("results_bundle.zip") and os.environ.get("HF_TOKEN"):
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.upload_file(
        path_or_fileobj="results_bundle.zip",
        path_in_repo="results_bundle.zip",
        repo_id=HF_REPO,
        repo_type="model",
        token=os.environ["HF_TOKEN"]
    )
    print(f"✅ [HF-SYNC] Uploaded 'results_bundle.zip' to https://huggingface.co/{HF_REPO}")